### Small Files

![](https://delta.io/_astro/image1.Cjq8K5tu_ev2TH.webp)

Este problema ocurre cuando los datos se dividen en miles de archivos de pocos KB. Spark pierde más tiempo abriendo/cerrando metadatos que leyendo datos.

**Síntomas**

- Consultar una tabla lleva minutos solo en la fase de "listing files" (metadata overhead), aunque la cantidad total de datos sea pequeña.

- Saturación del NameNode o metastore.

**Soluciones:**

- `OPTIMIZE`: Agrupa los archivos pequeños en archivos más grandes (idealmente de 1GB).

- **Auto Optimize:** Habilitar `autoOptimize.optimizeWrite` y `autoOptimize.autoCompact` para que Databricks gestione el tamaño durante la escritura.

- **Frecuencia de VACUUM**: No resuelve el rendimiento de lectura, pero elimina archivos antiguos que ya no son necesarios tras un OPTIMIZE.

- **Usar `coalesce()` antes de escribir:** A diferencia de `repartition()`, `coalesce()` reduce el número de particiones sin provocar un Shuffle completo.

In [0]:
# Reduce a 4 particiones antes de guardar
df.coalesce(4).write.format("delta").save(...)

### Data Skew

Ocurre cuando la carga de datos no se distribuye equitativamente entre las particiones. En un `JOIN` o `GROUP BY`, una clave específica (por ejemplo, `user_id = NULL` o un valor genérico) concentra el 80% de los registros en un solo Task, mientras los demás ejecutores terminan en 5 segundos y se quedan esperando al ejecutor "pesado".

- **Síntoma:** Durante un Join o Agregación, un solo "Stage" de Spark tarda mucho más que el resto, o un ejecutor se queda sin memoria (OOM) mientras los demás están inactivos.

**Soluciones:**

- **Salting (Técnica clásica):** Si haces un JOIN por una clave muy repetida, le agregas un sufijo aleatorio (un "salero", ej. 0 al 3) a la clave en la tabla izquierda y duplicas los registros en la tabla derecha con esos sufijos para repartir el peso entre múltiples particiones.

- **AQE Skew Join (Automatizado en Spark 3+):** Habilita Adaptive Query Execution. Spark detecta las particiones sesgadas en tiempo de ejecución y las divide automáticamente en subparticiones más pequeñas.

In [0]:
%sql
SET spark.sql.adaptive.enabled = true;
SET spark.sql.adaptive.skewJoin.enabled = true;

### Data Shuffling

Es el proceso de mover datos entre nodos del clúster a través de la red. Es la operación más cara en términos de tiempo.

- **Síntoma:** Alto uso de red y lentitud en operaciones de `JOIN`, `GROUP BY` o `DISTINCT`.

**Soluciones:**

- **Broadcast Hash Join:** Si una de las tablas es pequeña (menor a 10MB por defecto), Spark la copia en todos los nodos, evitando el shuffle de la tabla grande.

- **Z-Order Clustering:** Organiza los datos físicamente para que las columnas relacionadas estén en los mismos archivos, mejorando el "Data Skipping".

- **Evitar particionado excesivo:** No particionar por columnas con alta cardinalidad (muchos valores únicos), ya que genera demasiadas carpetas y archivos pequeños.

### Spill To Disk

Ocurre cuando una partición o dataset intermedio es demasiado grande para caber en la memoria RAM asignada al ejecutor (`spark.memory.fraction`). Para no estrellarse con un OOM, Spark "derrama" (spills) los datos temporales al disco local del nodo (RAM → Disk) y luego los vuelve a leer.

- **Síntoma:** En la UI de Spark verás una advertencia de "Spill (Memory)" y "Spill (Disk)". El rendimiento cae drásticamente (el disco es mucho más lento que la RAM).

**Soluciones:**

- **Filtrado temprano:** Aplicar `WHERE` y `SELECT` (solo las columnas necesarias) lo antes posible para reducir el volumen de datos en memoria.

- **Aumentar tamaño de instancia:** Usar Workers con más memoria RAM.

- **Reducir el tamaño de las particiones:** Aumentar `spark.sql.shuffle.partitions` para que cada tarea individual maneje menos datos.

- **AQE Auto-Optimizing:** Permite que Spark redimensione las particiones dinámicamente:

In [0]:
%sql
SET spark.sql.adaptive.coalescePartitions.enabled = true;

### **Out of Memory (OOM) Errors**

**¿Qué es?**

El proceso de Spark colapsa porque se sobrepasa la memoria física disponible en el Driver o en los Executors.

**Tipos y Soluciones**

**A. Driver OOM (`java.lang.OutOfMemoryError`: Java heap space)**
- **Causa:** Traer demasiados datos al nodo central usando `.collect()`, `.toPandas()`, o plots locales sobre datasets masivos.

- **Solución:** Evita `.collect()` en producción. Si necesitas exportar a pandas, filtra/agrega los datos en Spark primero. Si es indispensable, aumenta `spark.driver.memory`.

**B. Executor OOM**
- **Causa:** Particiones gigantescas (relacionado con Skew o Spill), o uso de funciones UDFs en Python no optimizadas que saturan la memoria del proceso Python/JVM.

- **Solución:** Usa Vectorized UDFs (PySpark Pandas UDFs) basadas en Apache Arrow en lugar de UDFs tradicionales de Python.

### Metadata Bottlenecks

Sucede cuando Spark tarda demasiado en listar los archivos en el almacenamiento de nube antes de empezar a procesar.

- **Síntoma:** La consulta tarda varios segundos o minutos en "arrancar" antes de que veas progreso en los Jobs.

**Soluciones:**

- **Delta Lake:** Al usar tablas Delta, Spark lee el _delta_log en lugar de listar archivos en S3/ADLS, lo que elimina este problema.

- **Partition Pruning:** Asegurarse de filtrar por la columna de partición en la cláusula `WHERE` para que Spark ignore carpetas enteras.